In [53]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_selection import RFECV, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list


In [54]:
# 设置random_state
random_state = 0

### feature selection step 1: ICC calculation for radimoics features from reader 1 and reader 2

In [55]:
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader2 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized_reader2.xlsx')

# we only keep the rows in df_reader1 that are also in df_reader2 based on Patient_set and Patient_index
df_reader1 = df_reader1[df_reader1['Patient_index'].isin(df_reader2['Patient_index']) & df_reader1['Patient_set'].isin(df_reader2['Patient_set'])]
print(f'Number of cases in reader 1 after matching: {len(df_reader1)}')

non_feature_cols = ['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath']
feature_cols = [col for col in df_reader1.columns if col not in non_feature_cols]

# we need to calculate the ICC for each feature between reader 1 and reader 2, if it's >0.75, we keep it
# calculate ICC, import packages 

icc_rows = []
for f in feature_cols:
    x = df_reader1[f].values
    y = df_reader2[f].values
    icc = ff.icc2_1(x, y)
    if icc<0.75:
        print('feature:', f, ' ICC:', icc)
    icc_rows.append({'Feature': f, 'ICC': icc})

# only keep features with ICC > 0.75
icc_df = pd.DataFrame(icc_rows)
selected_features = icc_df[icc_df['ICC'] > 0.75]['Feature'].tolist()
print('original number of features:', len(feature_cols))
print(f'Number of features with ICC > 0.75: {len(selected_features)}')

# dropped features
dropped_features = icc_df[icc_df['ICC'] <= 0.75]['Feature'].tolist()
# save dropped features to excel, file name: dropped_features.xlsx, sheet_name: 'inter_reader_icc'
dropped_df = pd.DataFrame({'dropped_feature': dropped_features})

with pd.ExcelWriter('/host/d/projects/Habitats/radiomics/whole_image/dropped_features.xlsx', engine='openpyxl') as writer:
    dropped_df.to_excel(writer, sheet_name='inter_reader_icc', index=False)

# now we create "df" for reader 1 with only selected features
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader1_selected = df_reader1[['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath'] + selected_features]
print('Shape of df_reader1_selected:', df_reader1_selected.shape)


Number of cases in reader 1 after matching: 28
feature: wavelet-LLH_glszm_GrayLevelNonUniformityNormalized  ICC: 0.6263630706816321
feature: wavelet-LLH_glszm_LargeAreaEmphasis  ICC: 0.5329332740187652
feature: wavelet-LLH_glszm_LargeAreaLowGrayLevelEmphasis  ICC: 0.3768460239335729
feature: wavelet-LLH_glszm_ZoneEntropy  ICC: 0.7283365041657534
feature: wavelet-LLH_glszm_ZoneVariance  ICC: 0.543677394394188
feature: wavelet-LLH_ngtdm_Busyness  ICC: 0.4740578781919682
feature: wavelet-LHH_glrlm_LongRunLowGrayLevelEmphasis  ICC: 0.6917611185313896
feature: wavelet-LHH_glszm_LargeAreaLowGrayLevelEmphasis  ICC: 0.5612562639490835
feature: wavelet-LHH_ngtdm_Busyness  ICC: 0.31184178856412614
feature: wavelet-HLH_glcm_ClusterShade  ICC: -0.08047851071601521
feature: wavelet-HLH_ngtdm_Busyness  ICC: 0.5524725353820513
original number of features: 1106
Number of features with ICC > 0.75: 1095
Shape of df_reader1_selected: (330, 1099)


### feature selection step 2: PCC


In [56]:
df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')

non_feature_cols = ["Patient_set","Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in df.columns if c not in non_feature_cols]

X = df[feature_cols].copy()

corr = X.corr(method='pearson').abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# 5) 找到需要删除的列：如果该列与任何其他列相关性 > 阈值，就删它
threshold = 0.90
to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]

print(f"Total features: {len(feature_cols)}")
print(f"Dropped due to PCC > {threshold}: {len(to_drop)}")
print(f"Remaining: {len(feature_cols) - len(to_drop)}")


Total features: 1106
Dropped due to PCC > 0.9: 821
Remaining: 285


In [57]:
X_selected = X.drop(columns=to_drop)

df_pcc = pd.concat([df[non_feature_cols], X_selected], axis=1)
df_pcc.to_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx', index=False)

# save dropped features to excel, file name: dropped_features.xlsx, sheet_name: 'pcc'
dropped_df = pd.DataFrame({'dropped_feature': to_drop})
with pd.ExcelWriter('/host/d/projects/Habitats/radiomics/whole_image/dropped_features.xlsx', engine='openpyxl', mode='a') as writer:
    dropped_df.to_excel(writer, sheet_name='pcc', index=False)


### Feature selection step 3: LASSO

In [58]:
radiomics_df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx')
label_df = pd.read_excel('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx')

# 检验radiomics_df和label_df的Patient_index是否完全匹配，如果不匹配，输出不匹配的Patient_index
radiomics_patient_index = set(radiomics_df['Patient_index'])
label_patient_index = set(label_df['Patient_index'])
if radiomics_patient_index != label_patient_index:
    print('不匹配的Patient_index:', radiomics_patient_index.symmetric_difference(label_patient_index))
else:
    print('Patient_index完全匹配')


non_feature_cols = ["Patient_set","Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]

y_col = 'Prognosis_label'


X = radiomics_df[feature_cols].values
y = label_df[y_col].values

print(f'Feature matrix shape: {X.shape}', f'Label vector shape: {y.shape}')


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LogisticRegressionCV(
        Cs=30, cv=cv, penalty="l1", solver="liblinear",
        scoring="roc_auc", max_iter=5000, refit=True, n_jobs=-1, random_state=random_state
    ))
])

model.fit(X, y)
lasso_cv = model.named_steps["lasso"]


Patient_index完全匹配
Feature matrix shape: (330, 285) Label vector shape: (330,)


In [60]:
print("Best C (1/lambda):", lasso_cv.C_[0])
print("Best mean CV AUC:", lasso_cv.scores_[1].mean(axis=0).max())

top_k = 20

coef = lasso_cv.coef_.ravel()
abs_coef = np.abs(coef)

nonzero_idx = np.where(coef != 0)[0]
print("Total features:", len(feature_cols))
print("Selected by LASSO non-zero:", len(nonzero_idx))

# First select non-zero coefficients ranked by absolute coefficient.
nonzero_sorted_idx = nonzero_idx[np.argsort(abs_coef[nonzero_idx])[::-1]]

if len(nonzero_sorted_idx) >= top_k:
    selected_idx = nonzero_sorted_idx[:top_k]
else:
    # If LASSO gives fewer than top_k non-zero features,
    # fill remaining slots using the largest absolute coefficients among all features.
    remaining_idx = [i for i in np.argsort(abs_coef)[::-1] if i not in set(nonzero_sorted_idx)]
    selected_idx = np.array(list(nonzero_sorted_idx) + remaining_idx[:top_k - len(nonzero_sorted_idx)])

selected_features = [feature_cols[i] for i in selected_idx]
selected_coef = coef[selected_idx]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f, c in zip(selected_features, selected_coef):
    print(f"  {f}: coef={c:.6f}")

# save radiomics features selected by LASSO top20
df_lasso = radiomics_df[non_feature_cols + selected_features]
df_lasso.to_excel(
    '/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_lasso_selected.xlsx',
    index=False
)


print("Saved selected feature table: radiomics_measurements_lasso_selected.xlsx")

Best C (1/lambda): 17.43328822199987
Best mean CV AUC: 0.5519867082136424
Total features: 285
Selected by LASSO non-zero: 170
Selected top K = 20
Selected features: 20
  wavelet-HHH_glcm_DifferenceEntropy: coef=-9.840232
  log-sigma-2-0-mm-3D_firstorder_90Percentile: coef=9.703559
  log-sigma-2-0-mm-3D_glszm_SizeZoneNonUniformity: coef=-7.391102
  log-sigma-4-0-mm-3D_glcm_InverseVariance: coef=7.165625
  wavelet-HLL_glcm_Correlation: coef=6.824621
  wavelet-HHL_firstorder_Maximum: coef=6.586404
  wavelet-HHH_glszm_GrayLevelNonUniformity: coef=-6.182304
  wavelet-HHL_glszm_GrayLevelNonUniformity: coef=5.804779
  wavelet-LLH_glszm_SizeZoneNonUniformityNormalized: coef=5.686660
  wavelet-LLH_glszm_GrayLevelNonUniformityNormalized: coef=5.142160
  log-sigma-4-0-mm-3D_glszm_SizeZoneNonUniformityNormalized: coef=-5.012329
  log-sigma-4-0-mm-3D_glrlm_RunEntropy: coef=4.743276
  log-sigma-6-0-mm-3D_glrlm_LongRunLowGrayLevelEmphasis: coef=4.593747
  wavelet-HHL_glszm_SizeZoneNonUniformity: coef

### Feature selection step 3B: RFE or SFS for different ML models

In [61]:
radiomics_df = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx')
label_df = pd.read_excel('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx')

# 检验radiomics_df和label_df的Patient_index是否完全匹配，如果不匹配，输出不匹配的Patient_index
radiomics_patient_index = set(radiomics_df['Patient_index'])
label_patient_index = set(label_df['Patient_index'])
if radiomics_patient_index != label_patient_index:
    print('不匹配的Patient_index:', radiomics_patient_index.symmetric_difference(label_patient_index))
else:
    print('Patient_index完全匹配')


non_feature_cols = ["Patient_set","Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]

y_col = 'Prognosis_label'


X = radiomics_df[feature_cols].values
y = label_df[y_col].values

print(f'Feature matrix shape: {X.shape}', f'Label vector shape: {y.shape}')


Patient_index完全匹配
Feature matrix shape: (330, 285) Label vector shape: (330,)


#### for SVM

#### SFS

In [43]:
random_state = 0
top_k = 20

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state
)

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="linear",
        C=1.0,
        class_weight="balanced",
        probability=True,
        random_state=random_state
    ))
])

sfs = SequentialFeatureSelector(
    estimator=svm_pipe,
    n_features_to_select=top_k,
    direction="forward",
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

sfs.fit(X, y)

support = sfs.get_support()

selected_features = [
    f for f, keep in zip(feature_cols, support) if keep
]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f in selected_features:
    print("  ", f)

out_dir = '/host/d/projects/Habitats/radiomics/whole_image'
os.makedirs(out_dir, exist_ok=True)

selected_out_path = os.path.join(
    out_dir,
    'radiomics_measurements_SVM_SFS_selected.xlsx'
)

df_selected = radiomics_df[non_feature_cols + selected_features].copy()
df_selected.to_excel(selected_out_path, index=False)


print("Saved selected feature table:", selected_out_path)


Selected top K = 20
Selected features: 20
   log-sigma-2-0-mm-3D_firstorder_Minimum
   log-sigma-2-0-mm-3D_glcm_ClusterShade
   log-sigma-2-0-mm-3D_glrlm_LongRunHighGrayLevelEmphasis
   log-sigma-2-0-mm-3D_glrlm_LongRunLowGrayLevelEmphasis
   log-sigma-4-0-mm-3D_glcm_Idn
   log-sigma-4-0-mm-3D_glrlm_LongRunHighGrayLevelEmphasis
   log-sigma-4-0-mm-3D_glszm_GrayLevelVariance
   log-sigma-4-0-mm-3D_gldm_SmallDependenceLowGrayLevelEmphasis
   wavelet-LLH_firstorder_Kurtosis
   wavelet-LLH_glszm_LowGrayLevelZoneEmphasis
   wavelet-LHH_firstorder_Entropy
   wavelet-HLL_firstorder_Kurtosis
   wavelet-HLL_glrlm_LongRunLowGrayLevelEmphasis
   wavelet-HLH_firstorder_Entropy
   wavelet-HLH_glcm_ClusterTendency
   wavelet-HLH_glcm_Contrast
   wavelet-HLH_glcm_Imc2
   wavelet-HHL_glszm_GrayLevelNonUniformityNormalized
   wavelet-HHH_glcm_ClusterShade
   wavelet-HHH_glszm_ZoneEntropy
Saved selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_SVM_SFS_selecte

### RFE

In [62]:
random_state = 0
top_k = 20

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="linear",
        C=1.0,
        class_weight="balanced",
        random_state=random_state
    ))
])

rfe = RFE(
    estimator=svm_pipe,
    n_features_to_select=top_k,
    step=1,
    importance_getter="named_steps.clf.coef_"
)

rfe.fit(X, y)

support = rfe.support_
ranking = rfe.ranking_

selected_features = [
    f for f, keep in zip(feature_cols, support) if keep
]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f in selected_features:
    print("  ", f)

out_dir = '/host/d/projects/Habitats/radiomics/whole_image'
os.makedirs(out_dir, exist_ok=True)

selected_out_path = os.path.join(
    out_dir,
    'radiomics_measurements_SVM_selected.xlsx'
)


df_selected = radiomics_df[non_feature_cols + selected_features].copy()
df_selected.to_excel(selected_out_path, index=False)

rank_df = pd.DataFrame({
    "feature": feature_cols,
    "ranking": ranking,
    "selected": support
}).sort_values(["ranking", "feature"])


print("Saved selected feature table:", selected_out_path)


Selected top K = 20
Selected features: 20
   original_firstorder_10Percentile
   original_ngtdm_Complexity
   log-sigma-2-0-mm-3D_firstorder_90Percentile
   log-sigma-2-0-mm-3D_firstorder_Kurtosis
   log-sigma-2-0-mm-3D_glcm_ClusterShade
   log-sigma-2-0-mm-3D_glcm_Idmn
   log-sigma-2-0-mm-3D_ngtdm_Contrast
   log-sigma-4-0-mm-3D_glcm_InverseVariance
   log-sigma-6-0-mm-3D_glrlm_LongRunLowGrayLevelEmphasis
   log-sigma-6-0-mm-3D_glszm_SmallAreaLowGrayLevelEmphasis
   wavelet-LLH_glrlm_GrayLevelVariance
   wavelet-LLH_glszm_SizeZoneNonUniformityNormalized
   wavelet-LHH_gldm_DependenceNonUniformityNormalized
   wavelet-LHH_ngtdm_Strength
   wavelet-HLL_firstorder_Skewness
   wavelet-HHL_firstorder_Maximum
   wavelet-HHL_glcm_Correlation
   wavelet-HHL_glszm_GrayLevelNonUniformity
   wavelet-HHL_ngtdm_Strength
   wavelet-HHH_glszm_GrayLevelNonUniformity
Saved selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_SVM_selected.xlsx


### for XGBoost

In [63]:
random_state = 0
top_k = 20

scale_pos_weight = np.sum(y == 0) / np.sum(y == 1)
print("scale_pos_weight:", scale_pos_weight)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=random_state,

    # avoid xgboost/joblib parallel crash in docker
    n_jobs=1,

    scale_pos_weight=scale_pos_weight,

    # fixed conservative values for feature selection
    n_estimators=100,
    max_depth=5,)


rfe = RFE(
    estimator=xgb,
    n_features_to_select=top_k,
    step=1
)

rfe.fit(X, y)


scale_pos_weight: 2.36734693877551


RFE(estimator=XGBClassifier(base_score=None, booster=None, callbacks=None,
                            colsample_bylevel=None, colsample_bynode=None,
                            colsample_bytree=None, device=None,
                            early_stopping_rounds=None,
                            enable_categorical=False, eval_metric='auc',
                            feature_types=None, feature_weights=None,
                            gamma=None, grow_policy=None, importance_type=None,
                            interaction_constraints=None, learning_rate=None,
                            max_bin=None, max_cat_threshold=None,
                            max_cat_to_onehot=None, max_delta_step=None,
                            max_depth=5, max_leaves=None, min_child_weight=None,
                            missing=nan, monotone_constraints=None,
                            multi_strategy=None, n_estimators=100, n_jobs=1,
                            num_parallel_tree=None, ...),
    n_features_to_select=20)

In [65]:
support = rfe.support_
ranking = rfe.ranking_

selected_features = [
    f for f, keep in zip(feature_cols, support) if keep
]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f in selected_features:
    print("  ", f)

out_dir = '/host/d/projects/Habitats/radiomics/whole_image'
os.makedirs(out_dir, exist_ok=True)

selected_out_path = os.path.join(
    out_dir,
    'radiomics_measurements_XGBoost_selected.xlsx'
)

df_selected = radiomics_df[non_feature_cols + selected_features].copy()
df_selected.to_excel(selected_out_path, index=False)

rank_df = pd.DataFrame({
    "feature": feature_cols,
    "ranking": ranking,
    "selected": support
}).sort_values(["ranking", "feature"])


print("Saved selected feature table:", selected_out_path)


Selected top K = 20
Selected features: 20
   original_shape_Elongation
   original_shape_SurfaceVolumeRatio
   original_firstorder_Skewness
   original_gldm_SmallDependenceLowGrayLevelEmphasis
   log-sigma-2-0-mm-3D_glcm_Idmn
   log-sigma-2-0-mm-3D_glrlm_RunEntropy
   log-sigma-2-0-mm-3D_glszm_ZoneEntropy
   log-sigma-4-0-mm-3D_gldm_SmallDependenceLowGrayLevelEmphasis
   log-sigma-6-0-mm-3D_glcm_Correlation
   log-sigma-6-0-mm-3D_glszm_SizeZoneNonUniformityNormalized
   wavelet-LLH_glszm_LargeAreaHighGrayLevelEmphasis
   wavelet-LHH_ngtdm_Busyness
   wavelet-HLL_firstorder_Skewness
   wavelet-HLL_glcm_Imc1
   wavelet-HLL_glcm_Idmn
   wavelet-HLL_glrlm_LongRunHighGrayLevelEmphasis
   wavelet-HHL_glcm_Correlation
   wavelet-HHH_glcm_Contrast
   wavelet-HHH_glcm_Imc2
   wavelet-HHH_ngtdm_Strength
Saved selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_XGBoost_selected.xlsx


### for Random forest

In [66]:
random_state = 0
top_k = 20

rf_selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=random_state,

    # avoid nested parallel issues
    n_jobs=1
)

rfe = RFE(
    estimator=rf_selector,
    n_features_to_select=top_k,
    step=1
)

rfe.fit(X, y)


RFE(estimator=RandomForestClassifier(class_weight='balanced',
                                     min_samples_leaf=3, n_estimators=300,
                                     n_jobs=1, random_state=0),
    n_features_to_select=20)

In [67]:
support = rfe.support_
ranking = rfe.ranking_

selected_features = [
    f for f, keep in zip(feature_cols, support) if keep
]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f in selected_features:
    print("  ", f)

out_dir = '/host/d/projects/Habitats/radiomics/whole_image'
os.makedirs(out_dir, exist_ok=True)

selected_out_path = os.path.join(
    out_dir,
    'radiomics_measurements_RF_selected.xlsx'
)

df_selected = radiomics_df[non_feature_cols + selected_features].copy()
df_selected.to_excel(selected_out_path, index=False)

rank_df = pd.DataFrame({
    "feature": feature_cols,
    "ranking": ranking,
    "selected": support
}).sort_values(["ranking", "feature"])

print("Saved selected feature table:", selected_out_path)


Selected top K = 20
Selected features: 20
   original_glcm_Correlation
   original_glcm_MaximumProbability
   original_gldm_SmallDependenceLowGrayLevelEmphasis
   log-sigma-2-0-mm-3D_firstorder_Mean
   log-sigma-2-0-mm-3D_firstorder_Median
   log-sigma-2-0-mm-3D_firstorder_RootMeanSquared
   log-sigma-2-0-mm-3D_glszm_ZoneEntropy
   log-sigma-2-0-mm-3D_gldm_DependenceEntropy
   log-sigma-4-0-mm-3D_firstorder_Maximum
   log-sigma-6-0-mm-3D_glcm_Correlation
   wavelet-LHL_glrlm_LongRunHighGrayLevelEmphasis
   wavelet-HLL_firstorder_Skewness
   wavelet-HLL_glcm_ClusterShade
   wavelet-HLL_glcm_Correlation
   wavelet-HLL_glcm_Imc1
   wavelet-HLL_glcm_Idmn
   wavelet-HLH_glszm_GrayLevelNonUniformityNormalized
   wavelet-HHL_ngtdm_Strength
   wavelet-HHH_glcm_Contrast
   wavelet-HHH_glrlm_ShortRunEmphasis
Saved selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_RF_selected.xlsx


## for KNN

In [68]:
random_state = 0
top_k = 20

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state
)

knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
    ))
])

sfs = SequentialFeatureSelector(
    estimator=knn_pipe,
    n_features_to_select=top_k,
    direction="forward",
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

sfs.fit(X, y)


SequentialFeatureSelector(cv=StratifiedKFold(n_splits=5, random_state=0, shuffle=True),
                          estimator=Pipeline(steps=[('scaler',
                                                     StandardScaler()),
                                                    ('clf',
                                                     KNeighborsClassifier(weights='distance'))]),
                          n_features_to_select=20, n_jobs=-1,
                          scoring='roc_auc')

In [69]:
support = sfs.get_support()

selected_features = [
    f for f, keep in zip(feature_cols, support) if keep
]

print("Selected top K =", top_k)
print("Selected features:", len(selected_features))
for f in selected_features:
    print("  ", f)

out_dir = '/host/d/projects/Habitats/radiomics/whole_image'
os.makedirs(out_dir, exist_ok=True)

selected_out_path = os.path.join(
    out_dir,
    'radiomics_measurements_KNN_SFS_selected.xlsx'
)


df_selected = radiomics_df[non_feature_cols + selected_features].copy()
df_selected.to_excel(selected_out_path, index=False)


print("Saved selected feature table:", selected_out_path)


Selected top K = 20
Selected features: 20
   original_glszm_GrayLevelNonUniformity
   log-sigma-2-0-mm-3D_glcm_Idmn
   wavelet-LLH_firstorder_Kurtosis
   wavelet-LLH_firstorder_Minimum
   wavelet-LLH_firstorder_Variance
   wavelet-LLH_glcm_ClusterProminence
   wavelet-LLH_glszm_GrayLevelVariance
   wavelet-LLH_glszm_SizeZoneNonUniformity
   wavelet-LLH_ngtdm_Complexity
   wavelet-LLH_ngtdm_Strength
   wavelet-LHH_firstorder_Entropy
   wavelet-LHH_firstorder_Kurtosis
   wavelet-LHH_glcm_ClusterProminence
   wavelet-LHH_glcm_ClusterShade
   wavelet-LHH_glcm_Idmn
   wavelet-LHH_ngtdm_Busyness
   wavelet-LHH_ngtdm_Strength
   wavelet-HLH_firstorder_Kurtosis
   wavelet-HHH_firstorder_RootMeanSquared
   wavelet-HHH_glszm_GrayLevelNonUniformity
Saved selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_KNN_SFS_selected.xlsx
